<a href="https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
import os
import subprocess
import numpy as np
import pandas as pd

REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

# Clone starter repo if it is not already present
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

data_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Used only to audit the two signals.
# It is NOT used as an input to the baseline.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# ---------------------------------------------------------
# SIGNAL 1: STALENESS
# ---------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, np.inf],
    labels=["0-90d", "91-180d", "181d+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

staleness_table["declining_rate"] = (
    staleness_table["declining_rate"].round(3)
)

print("\nSIGNAL 1 — STALENESS")
display(staleness_table)

stale_change = (
    staleness_table["declining_rate"].iloc[-1]
    - staleness_table["declining_rate"].iloc[0]
)

if stale_change > 0.05:
    print("Verdict: CONFIRMED")
elif stale_change < -0.05:
    print("Verdict: OPPOSITE")
else:
    print("Verdict: MIXED")

# ---------------------------------------------------------
# SIGNAL 2: SEARCH VISIBILITY
# ---------------------------------------------------------

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 100, 500, 3000, np.inf],
    labels=["0-100", "101-500", "501-3000", "3000+"]
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

visibility_table["declining_rate"] = (
    visibility_table["declining_rate"].round(3)
)

print("\nSIGNAL 2 — SEARCH VISIBILITY")
display(visibility_table)

visibility_change = (
    visibility_table["declining_rate"].iloc[-1]
    - visibility_table["declining_rate"].iloc[0]
)

if visibility_change > 0.05:
    print("Verdict: CONFIRMED")
elif visibility_change < -0.05:
    print("Verdict: OPPOSITE")
else:
    print("Verdict: MIXED")

print(
    "\nThe declining label is used only for signal auditing; "
    "it is not an input to the baseline score."
)

Rows: 30000
Columns: 44

SIGNAL 1 — STALENESS


,staleness_bucket,n,declining_rate
0,0-90d,20655,0.512
1,91-180d,9171,0.611
2,181d+,174,0.471


Verdict: MIXED

SIGNAL 2 — SEARCH VISIBILITY


,visibility_bucket,n,declining_rate
0,0-100,8006,0.389
1,101-500,5279,0.604
2,501-3000,8432,0.621
3,3000+,8283,0.570


Verdict: CONFIRMED

The declining label is used only for signal auditing; it is not an input to the baseline score.


My baseline rule is to prioritize stale pages that still have meaningful search visibility. A page is considered stale when it has not been updated for at least 180 days, and visible when it has at least 500 impressions in the last 90 days. The baseline score increases with impressions for pages that satisfy both conditions, so highly visible stale pages are ranked first for refresh review.

Reason codes:
- stale_visible: page is stale and has enough recent visibility to justify refresh review.
- stale_low_visibility: page is stale but has limited recent visibility.
- fresh_visible: page has visibility but is not stale.
- other: page does not meet the stronger stale-and-visible condition.

The two signals behind this rule are staleness and search visibility. Staleness is directly linked to the refresh-review logic discussed in the session; visibility is used to distinguish pages that still have meaningful search exposure from pages with little current exposure.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One transparent baseline rule:
# prioritize pages that are both stale and visible.

stale = (
    df["days_since_last_update"] >= 180
).astype(int)

visible = (
    df["impressions_90d"] >= 500
).astype(int)

# Score:
# stale + visible pages get a score equal to their impressions.
# All other pages receive 0.
df["baseline_action_score"] = (
    stale * visible * df["impressions_90d"]
)

# One reason code per row
df["reason_code"] = np.select(
    [
        (stale == 1) & (visible == 1),
        (stale == 1) & (visible == 0),
        (stale == 0) & (visible == 1),
    ],
    [
        "stale_visible",
        "stale_low_visibility",
        "fresh_visible",
    ],
    default="other"
)

# Action supported by the rule
df["action_label"] = np.where(
    df["baseline_action_score"] > 0,
    "refresh_review",
    "monitor"
)

# Rank every row, highest score first
df["baseline_rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Required ranked queue
queue_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

baseline_queue = (
    df[queue_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)

# Write required CSV
output_path = os.path.join(
    REPO_DIR,
    "work",
    "outputs",
    "baseline_action_score.csv"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

baseline_queue.to_csv(
    output_path,
    index=False
)

print("Baseline rule: stale AND visible")
print("Stale threshold: 180 days")
print("Visibility threshold: 500 impressions in 90 days")
print("Rows in ranked queue:", len(baseline_queue))
print("CSV written to:", output_path)

display(baseline_queue.head(10))


Baseline rule: stale AND visible
Stale threshold: 180 days
Visibility threshold: 500 impressions in 90 days
Rows in ranked queue: 30000
CSV written to: flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr
0,content_cf56e2e2e282,client_7f2253d7e2,1,61678,stale_visible,refresh_review,61678,194,19.7,0.15
1,content_7368877ea310,client_7f2253d7e2,2,59472,stale_visible,refresh_review,59472,194,24.8,0.13
2,content_1bfaa38ff26c,client_7f2253d7e2,3,25715,stale_visible,refresh_review,25715,194,22.2,0.23
3,content_0a91db491d14,client_7f2253d7e2,4,13299,stale_visible,refresh_review,13299,193,10.5,0.49
4,content_5feee3994adb,client_7f2253d7e2,5,7812,stale_visible,refresh_review,7812,194,39.0,0.01
5,content_c2d929d83eaa,client_7f2253d7e2,6,7558,stale_visible,refresh_review,7558,193,17.9,0.20
6,content_b16bd7307b39,client_7f2253d7e2,7,4590,stale_visible,refresh_review,4590,194,31.0,0.00
7,content_fe16a55cd13d,client_7f2253d7e2,8,4556,stale_visible,refresh_review,4556,194,16.4,0.33
8,content_ecb6215e79fd,client_7f2253d7e2,9,4429,stale_visible,refresh_review,4429,194,25.3,0.38
9,content_928af3e22c80,client_7f2253d7e2,10,1697,stale_visible,refresh_review,1697,193,15.8,0.12


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the top 10 recommendations
top10 = baseline_queue.head(10).copy()

print("TOP-10 REVIEW\n")

for _, row in top10.iterrows():

    if row["reason_code"] == "stale_visible":
        confidence = (
            "High: the page satisfies both baseline conditions "
            "(180+ days since update and 500+ impressions)."
        )

        wrong = (
            "The page could be a poor refresh choice if the high visibility "
            "is temporary, the update date is incomplete, or another page "
            "has a stronger business reason for review."
        )

    elif row["reason_code"] == "stale_low_visibility":
        confidence = (
            "Medium: the page is stale but does not have strong visibility."
        )

        wrong = (
            "Low visibility may mean that limited editorial capacity "
            "should be spent on a different page."
        )

    elif row["reason_code"] == "fresh_visible":
        confidence = (
            "Low: the page has visibility but does not meet the staleness rule."
        )

        wrong = (
            "A visible but recently updated page may not need an immediate refresh."
        )

    else:
        confidence = (
            "Low: the page does not satisfy the strongest baseline conditions."
        )

        wrong = (
            "A stronger stale-and-visible candidate may be more appropriate "
            "for the limited review queue."
        )

    print(
        f"Rank {int(row['baseline_rank'])}: "
        f"{row['content_id']}\n"
        f"  Action: {row['action_label']}\n"
        f"  Reason: {row['reason_code']}\n"
        f"  Confidence: {confidence}\n"
        f"  What would make it wrong: {wrong}\n"
    )


TOP-10 REVIEW

Rank 1: content_cf56e2e2e282
  Action: refresh_review
  Reason: stale_visible
  Confidence: High: the page satisfies both baseline conditions (180+ days since update and 500+ impressions).
  What would make it wrong: The page could be a poor refresh choice if the high visibility is temporary, the update date is incomplete, or another page has a stronger business reason for review.

Rank 2: content_7368877ea310
  Action: refresh_review
  Reason: stale_visible
  Confidence: High: the page satisfies both baseline conditions (180+ days since update and 500+ impressions).
  What would make it wrong: The page could be a poor refresh choice if the high visibility is temporary, the update date is incomplete, or another page has a stronger business reason for review.

Rank 3: content_1bfaa38ff26c
  Action: refresh_review
  Reason: stale_visible
  Confidence: High: the page satisfies both baseline conditions (180+ days since update and 500+ impressions).
  What would make it wrong

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weaker recommendations are the pages near the bottom of the top-10 queue because they satisfy the same stale-and-visible rule but have less exposure than the strongest recommendations above them. I would treat these as decision-support rather than guaranteed refresh priorities because the rule does not account for business importance, temporary traffic changes, or other editorial context.

The baseline score uses only staleness and recent visibility. It does not use product decision flags, the observed declining label, future-window measurements, or label-derived fields. Therefore the rule is designed to avoid future leakage and to remain a transparent pre-decision baseline.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# WEAK PICKS + LEAKAGE CHECK
# ---------------------------------------------------------

# The last 3 rows within the top-10 are the weakest
# recommendations among the reviewed pages.
weak_picks = baseline_queue.head(10).tail(3)

print("WEAKEST PICKS WITHIN TOP-10\n")

for _, row in weak_picks.iterrows():
    print(
        f"Rank {int(row['baseline_rank'])}: "
        f"score={row['baseline_action_score']:.0f}, "
        f"reason={row['reason_code']}, "
        f"action={row['action_label']}, "
        f"impressions={row['impressions_90d']:.0f}, "
        f"days_since_update={row['days_since_last_update']:.0f}"
    )

    print(
        "  Why it may be weak: it is lower in the queue because "
        "its recent visibility is smaller than the strongest "
        "stale-and-visible recommendations."
    )
    print()

# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

# These are the only two inputs used by the baseline score.
baseline_inputs = {
    "days_since_last_update",
    "impressions_90d"
}

# Inputs that would create leakage or use product decisions.
forbidden_inputs = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "needs_ctr_fix",
    "needs_refresh",
    "quick_win"
}

used_forbidden_inputs = (
    baseline_inputs.intersection(forbidden_inputs)
)

print("Forbidden inputs used by baseline score:",
      used_forbidden_inputs)

assert len(used_forbidden_inputs) == 0

print(
    "Leakage check: PASS — baseline score uses only "
    "staleness and visibility."
)

print("Future-window inputs used: NO")
print("Product decision flags used: NO")
print("Label-derived inputs used: NO")


WEAKEST PICKS WITHIN TOP-10

Rank 8: score=4556, reason=stale_visible, action=refresh_review, impressions=4556, days_since_update=194
  Why it may be weak: it is lower in the queue because its recent visibility is smaller than the strongest stale-and-visible recommendations.

Rank 9: score=4429, reason=stale_visible, action=refresh_review, impressions=4429, days_since_update=194
  Why it may be weak: it is lower in the queue because its recent visibility is smaller than the strongest stale-and-visible recommendations.

Rank 10: score=1697, reason=stale_visible, action=refresh_review, impressions=1697, days_since_update=193
  Why it may be weak: it is lower in the queue because its recent visibility is smaller than the strongest stale-and-visible recommendations.

Forbidden inputs used by baseline score: set()
Leakage check: PASS — baseline score uses only staleness and visibility.
Future-window inputs used: NO
Product decision flags used: NO
Label-derived inputs used: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.